# 🧠 4-Phase Math Model Fine-Tuning Pipeline
## LLama 3.1 8B Instruct | Unsloth QLoRA | Google Colab A100

This notebook implements a 4-phase fine-tuning pipeline using the **Llama 3.1 conversational format**:
- **Phase 1a/1b/1c**: General math knowledge (2.75M problems split into 3 chunks of ~918K)
- **Phase 2**: Extended math knowledge (14M+ problems) — *Optional*
- **Phase 3**: Mathematical reasoning (99K problems with reasoning chains)
- **Phase 4**: Tool-calling (Calculator + Math Cheatsheet RAG)

**Optimizations applied:**
- `packing = True` for efficient GPU utilization (no padding waste)
- `load_dataset("json")` for fast data loading
- NaN-safe training with `max_grad_norm` and gradient clipping
- Batch size 16 with gradient accumulation

---

## Cell 1 — Installation

In [ ]:
%%capture
!pip install unsloth
# Also get the latest nightly Unsloth!
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
# Additional dependencies
!pip install tensorboard


## Cell 2 — Mount Google Drive & Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ============================================================
# ⚙️  MASTER CONFIGURATION — EDIT THIS SECTION
# ============================================================

# ---- Phase Selection ----
# Phase 1 is split into 3 sub-phases (1a, 1b, 1c) for Colab session safety.
# Set to: "1a", "1b", "1c", 2, 3, or 4
CURRENT_PHASE = 3

# ---- Google Drive Paths ----
DRIVE_BASE = "/content/drive/MyDrive/SOTA_MATH"

# Dataset paths (all on Google Drive)
# Phase 1 is split into 3 chunks (~918K samples each)
DATASET_PATHS = {
    "1a": f"{DRIVE_BASE}/training_base_1.jsonl",
    "1b": f"{DRIVE_BASE}/training_base_2.jsonl",
    "1c": f"{DRIVE_BASE}/training_base_3.jsonl",
    2: f"{DRIVE_BASE}/extended_math.jsonl",
    3: f"{DRIVE_BASE}/final_dataset.jsonl",
    4: f"{DRIVE_BASE}/tool_calling_dataset.jsonl",
}

# Phase label for directories (1a/1b/1c all save under phase_1)
PHASE_DIR_LABEL = CURRENT_PHASE if isinstance(CURRENT_PHASE, int) else f"1_{CURRENT_PHASE[-1]}"

# Output directories (all on Google Drive for persistence across sessions)
CHECKPOINT_DIR = f"{DRIVE_BASE}/checkpoints/phase_{PHASE_DIR_LABEL}"
MERGED_MODEL_DIR = f"{DRIVE_BASE}/merged_model/phase_{PHASE_DIR_LABEL}"
LORA_DIR = f"{DRIVE_BASE}/lora_adapters/phase_{PHASE_DIR_LABEL}"
LOG_DIR = f"{DRIVE_BASE}/logs/phase_{PHASE_DIR_LABEL}"

for d in [CHECKPOINT_DIR, MERGED_MODEL_DIR, LORA_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

# ---- Model Configuration ----
MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 4096
LOAD_IN_4BIT = True
DTYPE = None  # Auto-detect

# ---- LoRA Configuration ----
LORA_R = 64
LORA_ALPHA = 64

# ---- Training Hyperparameters per Phase ----
# Phase 1a/1b/1c: packing=True allows batch_size=16 since avg tokens ~350
PHASE_CONFIGS = {
    "1a": dict(num_train_epochs=1, per_device_train_batch_size=12,
              gradient_accumulation_steps=8, learning_rate=2e-4,
              lr_scheduler_type="cosine", warmup_steps=50, weight_decay=0.01),
    "1b": dict(num_train_epochs=1, per_device_train_batch_size=12,
              gradient_accumulation_steps=8, learning_rate=1e-4,
              lr_scheduler_type="cosine", warmup_steps=30, weight_decay=0.01),
    "1c": dict(num_train_epochs=1, per_device_train_batch_size=12,
              gradient_accumulation_steps=8, learning_rate=5e-5,
              lr_scheduler_type="cosine", warmup_steps=30, weight_decay=0.01),
    2: dict(num_train_epochs=1, per_device_train_batch_size=12,
            gradient_accumulation_steps=8, learning_rate=1e-4,
            lr_scheduler_type="cosine", warmup_steps=30, weight_decay=0.01),
    3: dict(num_train_epochs=2, per_device_train_batch_size=16,
            gradient_accumulation_steps=1, learning_rate=3e-5,
            lr_scheduler_type="cosine", warmup_steps=20, weight_decay=0.01),
    4: dict(num_train_epochs=3, per_device_train_batch_size=8,
            gradient_accumulation_steps=2, learning_rate=2e-5,
            lr_scheduler_type="linear", warmup_steps=50, weight_decay=0.01),
}

# ---- Hugging Face Hub ----
HF_REPO_ID = "MY_HF_REPO_ID"
HF_TOKEN = "MY_HF_TOKEN"

# ---- Logging ----
LOGGING_STEPS = 100
SAVE_STEPS = 5000
RESUME_FROM_CHECKPOINT = False

# ---- System prompts per phase ----
SYSTEM_PROMPTS = {
    "1a": "You are a highly skilled math expert. Solve the given problem step by step with a complete solution and provide the final answer.",
    "1b": "You are a highly skilled math expert. Solve the given problem step by step with a complete solution and provide the final answer.",
    "1c": "You are a highly skilled math expert. Solve the given problem step by step with a complete solution and provide the final answer.",
    2: "You are a highly skilled math expert. Solve the given problem step by step with a complete solution and provide the final answer.",
    3: ("You are a highly skilled math expert and reasoning specialist. For each problem:\n"
        "1. First, reason through the problem — identify what is being asked, what concepts apply, and outline your approach.\n"
        "2. Then, provide the detailed step-by-step solution.\n"
        "3. Finally, state the final answer clearly."),
    4: ("You are a highly skilled math expert with access to external tools. "
        "Use the available tools to verify calculations and look up concepts/formulas when needed.\n\n"
        "Available Tools:\n"
        '1. calculator — Evaluate a mathematical expression accurately.\n'
        '   Usage: <tool_call>{"name": "calculator", "arguments": {"expression": "YOUR_EXPR"}}</tool_call>\n'
        '   The system will respond with: <tool_response>{"result": "VALUE"}</tool_response>\n\n'
        '2. math_cheatsheet — Look up a mathematical concept, formula, or theorem.\n'
        '   Usage: <tool_call>{"name": "math_cheatsheet", "arguments": {"topic": "YOUR_TOPIC"}}</tool_call>\n'
        '   The system will respond with: <tool_response>{"content": "RELEVANT_INFO"}</tool_response>\n\n'
        "For each problem: reason through it, use tools as needed, provide the solution, and state the final answer."),
}

config = PHASE_CONFIGS[CURRENT_PHASE]
print(f"{'='*60}")
print(f"  🚀 Phase {CURRENT_PHASE} Configuration")
print(f"{'='*60}")
print(f"  Model:          {MODEL_NAME}")
print(f"  Dataset:        {DATASET_PATHS[CURRENT_PHASE]}")
print(f"  Epochs:         {config['num_train_epochs']}")
print(f"  Batch Size:     {config['per_device_train_batch_size']}")
print(f"  Grad Accum:     {config['gradient_accumulation_steps']}")
print(f"  Effective BS:   {config['per_device_train_batch_size'] * config['gradient_accumulation_steps']}")
print(f"  Learning Rate:  {config['learning_rate']}")
print(f"  LoRA r/alpha:   {LORA_R}/{LORA_ALPHA}")
print(f"  Packing:        True")
print(f"  Save Steps:     {SAVE_STEPS}")
print(f"{'='*60}")


## Cell 3 — Load Model & Apply LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

# Load from previous sub-phase if available
# Chain: base → 1a → 1b → 1c → 2 → 3 → 4
load_model = MODEL_NAME
PREV_PHASE_MAP = {"1a": None, "1b": "1a", "1c": "1b", 2: "1c", 3: "1a", 4: 3}
prev_phase = PREV_PHASE_MAP.get(CURRENT_PHASE)

if prev_phase is not None:
    prev_dir_label = prev_phase if isinstance(prev_phase, int) else f"1_{prev_phase[-1]}"
    prev_lora = f"{DRIVE_BASE}/lora_adapters/phase_{prev_dir_label}"
    if os.path.exists(prev_lora) and os.listdir(prev_lora):
        load_model = prev_lora
        print(f"✅ Loading Phase {prev_phase} LoRA checkpoint: {prev_lora}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = load_model,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = DTYPE,
    load_in_4bit = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)


## Cell 4 — Data Prep (Llama 3.1 Conversational Format)

We use the `llama-3.1` chat template via Unsloth's `get_chat_template`.

Our JSONL data is converted to the HuggingFace conversation format:
```
{"role": "system", "content": "..."}
{"role": "user", "content": "..."}
{"role": "assistant", "content": "..."}
```

Then `tokenizer.apply_chat_template` renders it into the Llama 3.1 format:
```
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a math expert...<|eot_id|><|start_header_id|>user<|end_header_id|>

Solve x + 2 = 5<|eot_id|><|start_header_id|>assistant<|end_header_id|>

x = 3<|eot_id|>
```

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }
pass

print("✅ Llama 3.1 chat template loaded")


## Cell 5 — Load JSONL Dataset & Convert to Conversations

Uses `load_dataset("json")` for fast, memory-efficient loading.
Then `.map()` converts each sample to conversation format.

In [ ]:
import time
from datasets import load_dataset

dataset_path = DATASET_PATHS[CURRENT_PHASE]
system_prompt = SYSTEM_PROMPTS[CURRENT_PHASE]

print(f"📂 Loading dataset: {dataset_path}")
if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Dataset not found at {dataset_path}")

start_time = time.time()

# ---- Fast loading with HuggingFace datasets ----
raw_dataset = load_dataset("json", data_files=dataset_path, split="train")
print(f"  Loaded {len(raw_dataset):,} samples in {time.time() - start_time:.1f}s")

# ---- Convert to conversation format using .map() ----
def convert_to_conversations(example):
    """Convert a single JSONL sample to Llama 3.1 conversation format."""
    question = str(example.get("question", "")).strip()
    solution = str(example.get("solution", "")).strip()
    answer = str(example.get("answer", "")).strip()

    # Skip invalid samples by returning empty conversation
    if CURRENT_PHASE in ["1a", "1b", "1c", 2, 3]:
        if not question or not solution:
            return {"conversations": [], "valid": False}
    elif CURRENT_PHASE == 4:
        if not example.get("conversations"):
            return {"conversations": [], "valid": False}

    # Build assistant response based on phase
    phase_key = CURRENT_PHASE
    if phase_key in ["1a", "1b", "1c", 2]:
        assistant_content = f"**Solution:**\n{solution}\n\n**Answer:** {answer}"
    elif phase_key == 3:
        reasoning = str(example.get("reasoning", "")).strip()
        if not reasoning:
            return {"conversations": [], "valid": False}
        assistant_content = (
            f"**Reasoning:**\n{reasoning}\n\n"
            f"**Solution:**\n{solution}\n\n"
            f"**Answer:** {answer}"
        )
    elif phase_key == 4:
        convo = example.get("conversations", [])
        if not convo or len(convo) < 3:
            return {"conversations": [], "valid": False}
        return {"conversations": convo, "valid": True}

    conversation = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
        {"role": "assistant", "content": assistant_content},
    ]
    return {"conversations": conversation, "valid": True}

print(f"  Converting to conversation format (Phase {CURRENT_PHASE})...")
dataset = raw_dataset.map(convert_to_conversations, num_proc=4,
                          desc="Converting to conversations")

# Filter out invalid samples
total_before = len(dataset)
dataset = dataset.filter(lambda x: x["valid"], num_proc=4)
skipped = total_before - len(dataset)
print(f"  Skipped: {skipped:,} invalid samples")

# Remove the helper column
dataset = dataset.remove_columns(["valid"])

# Shuffle
dataset = dataset.shuffle(seed=3407)

# Apply chat template formatting
print(f"\n📝 Applying Llama 3.1 chat template...")
dataset = dataset.map(formatting_prompts_func, batched=True, num_proc=4,
                      desc="Applying chat template")

elapsed = time.time() - start_time
print(f"\n✅ Dataset ready: {len(dataset):,} samples in {elapsed:.1f}s")
print(f"\n📋 Sample conversation (item 0):")
print(dataset[0]["conversations"])
print(f"\n📋 Formatted text preview:")
print(dataset[0]["text"][:600])
print("...")


## Cell 6 — Training Setup (Phase-Aware)

**Phase 1a/1b/1c/2**: `packing=True` for GPU efficiency (avg ~350 tokens/sample)

**Phase 3/4**: `packing=False` + `train_on_responses_only` for precise reasoning training
- Only trains on assistant responses (reasoning + solution + answer)
- Question/system tokens are masked from loss computation

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

config = PHASE_CONFIGS[CURRENT_PHASE]

# Phase 3 & 4: packing=False + train_on_responses_only for better reasoning
# Phase 1 & 2: packing=True for speed
USE_PACKING = CURRENT_PHASE not in [3, 4]

trainer_kwargs = dict(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 4,
    packing = USE_PACKING,
    args = TrainingArguments(
        num_train_epochs = config["num_train_epochs"],
        per_device_train_batch_size = config["per_device_train_batch_size"],
        gradient_accumulation_steps = config["gradient_accumulation_steps"],
        learning_rate = config["learning_rate"],
        lr_scheduler_type = config["lr_scheduler_type"],
        warmup_steps = config["warmup_steps"],
        weight_decay = config["weight_decay"],
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = LOGGING_STEPS,
        logging_dir = LOG_DIR,
        report_to = "tensorboard",
        logging_strategy = "steps",
        logging_first_step = True,
        save_steps = SAVE_STEPS,
        save_total_limit = 3,
        output_dir = CHECKPOINT_DIR,
        optim = "adamw_8bit",
        seed = 3407,
        max_grad_norm = 0.3,
    ),
)

# For packing=False, add DataCollatorForSeq2Seq
if not USE_PACKING:
    trainer_kwargs["data_collator"] = DataCollatorForSeq2Seq(tokenizer=tokenizer)

trainer = SFTTrainer(**trainer_kwargs)

# Phase 3 & 4: Apply train_on_responses_only
# This masks loss on system + user tokens, training ONLY on assistant responses
# (reasoning + solution + answer) for better reasoning quality
if not USE_PACKING:
    from unsloth.chat_templates import train_on_responses_only
    trainer = train_on_responses_only(
        trainer,
        instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
        response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
    )
    print("✅ train_on_responses_only enabled — training only on reasoning + answer")

print(f"\n✅ Trainer ready (packing={'ON' if USE_PACKING else 'OFF'})")
print(f"   Phase: {CURRENT_PHASE}")
print(f"   Effective batch size: {config['per_device_train_batch_size'] * config['gradient_accumulation_steps']}")
if USE_PACKING:
    print(f"   Estimated packed sequences: ~{len(dataset) * 350 // MAX_SEQ_LENGTH:,}")
else:
    print(f"   Total sequences: {len(dataset):,} (no packing — each sample is a separate sequence)")
    print(f"   Loss masked on: system + user tokens")
    print(f"   Training on: assistant responses only (reasoning + solution + answer)")


In [ ]:
# Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")


## Cell 7 — 🚀 Train!

In [ ]:
import datetime, json, time

print(f"🚀 Starting Phase {CURRENT_PHASE} — {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

train_log = {
    "phase": str(CURRENT_PHASE), "start_time": datetime.datetime.now().isoformat(),
    "config": config, "model": MODEL_NAME, "dataset_size": len(dataset),
    "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "packing": True,
}

trainer_stats = trainer.train(
    resume_from_checkpoint = RESUME_FROM_CHECKPOINT if RESUME_FROM_CHECKPOINT else None
)

# Save training log
train_log["end_time"] = datetime.datetime.now().isoformat()
train_log["final_loss"] = trainer_stats.training_loss
train_log["total_steps"] = trainer_stats.global_step
train_log["runtime_sec"] = trainer_stats.metrics['train_runtime']
with open(os.path.join(LOG_DIR, f"training_log_phase_{CURRENT_PHASE}.json"), "w") as f:
    json.dump(train_log, f, indent=2)

print(f"\n✅ Phase {CURRENT_PHASE} complete!")
print(f"  Loss: {trainer_stats.training_loss:.4f}")
print(f"  Steps: {trainer_stats.global_step:,}")
print(f"  Runtime: {trainer_stats.metrics['train_runtime']/3600:.2f} hours")


In [ ]:
# Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")


## Cell 8 — TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {LOG_DIR}


## Cell 9 — 🧪 Inference

Test the model using the Llama 3.1 conversational format.

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)
FastLanguageModel.for_inference(model)  # 2x faster inference

messages = [
    {"role": "system", "content": SYSTEM_PROMPTS[CURRENT_PHASE]},
    {"role": "user", "content": "Solve for x: 3x + 7 = 22"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 1024, use_cache = True,
                         temperature = 1.5, min_p = 0.1)
tokenizer.batch_decode(outputs)


In [ ]:
# Streaming inference with TextStreamer
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": SYSTEM_PROMPTS[CURRENT_PHASE]},
    {"role": "user", "content": "Find the derivative of f(x) = x^3 - 4x^2 + 2x - 1"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 1024,
                   use_cache = True, temperature = 1.5, min_p = 0.1)


## Cell 10 — Custom Inference

Change the question and run this cell to test.

In [ ]:
FastLanguageModel.for_inference(model)

your_question = "What is the probability of rolling two dice and getting a sum of 7?"

messages = [
    {"role": "system", "content": SYSTEM_PROMPTS[CURRENT_PHASE]},
    {"role": "user", "content": your_question},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize = True, add_generation_prompt = True, return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 1024,
                   use_cache = True, temperature = 1.5, min_p = 0.1)


## Cell 11 — Save LoRA + Full Merged Model to Google Drive

In [ ]:
# Save LoRA adapters
print(f"💾 Saving LoRA adapters to: {LORA_DIR}")
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f"  ✅ LoRA saved")

# Save full merged model (16-bit)
print(f"\n💾 Saving merged 16-bit model to: {MERGED_MODEL_DIR}")
model.save_pretrained_merged(MERGED_MODEL_DIR, tokenizer, save_method = "merged_16bit",)
print(f"  ✅ Merged model saved")


In [ ]:
import shutil, os

# Step 1: Save merged model to LOCAL disk (not Drive)
LOCAL_GGUF_DIR = f"/content/gguf_export"
os.makedirs(LOCAL_GGUF_DIR, exist_ok=True)

# Step 2: Convert to GGUF from local path
model.save_pretrained_gguf(
    LOCAL_GGUF_DIR, tokenizer,
    quantization_method=["q4_k_m", "f16", "q8_0"]
)
print("✅ All 3 GGUF formats saved")

# Step 3: Copy GGUF files to Google Drive
DRIVE_GGUF_DIR = f"{DRIVE_BASE}/exports/phase_{PHASE_DIR_LABEL}/gguf"
os.makedirs(DRIVE_GGUF_DIR, exist_ok=True)

for f in os.listdir(LOCAL_GGUF_DIR):
    if f.endswith(".gguf"):
        print(f"  Copying {f} to Drive...")
        shutil.copy2(os.path.join(LOCAL_GGUF_DIR, f), DRIVE_GGUF_DIR)

print(f"✅ GGUF files copied to {DRIVE_GGUF_DIR}")


In [ ]:
# Unsloth appends "_gguf" to the output directory
LOCAL_GGUF_ACTUAL = "/content/gguf_export_gguf"
DRIVE_GGUF_DIR = f"{DRIVE_BASE}/exports/phase_{PHASE_DIR_LABEL}/gguf"
os.makedirs(DRIVE_GGUF_DIR, exist_ok=True)

for f in os.listdir(LOCAL_GGUF_ACTUAL):
    if f.endswith(".gguf"):
        src = os.path.join(LOCAL_GGUF_ACTUAL, f)
        size_gb = os.path.getsize(src) / 1024**3
        print(f"  Copying {f} ({size_gb:.1f} GB) to Drive...")
        shutil.copy2(src, DRIVE_GGUF_DIR)

print(f"\n✅ Done! Files in {DRIVE_GGUF_DIR}:")
for f in os.listdir(DRIVE_GGUF_DIR):
    print(f"  {f}")


In [ ]:
# Step 1: Save merged 16-bit model to LOCAL disk
LOCAL_VLLM_DIR = "/content/vllm_export"
os.makedirs(LOCAL_VLLM_DIR, exist_ok=True)

print("💾 Saving vLLM-compatible merged 16-bit model locally...")
model.save_pretrained_merged(LOCAL_VLLM_DIR, tokenizer, save_method="merged_16bit")
print("✅ vLLM format saved locally")

# Step 2: Copy to Google Drive
DRIVE_VLLM_DIR = f"{DRIVE_BASE}/exports/phase_{PHASE_DIR_LABEL}/vllm"

# Remove old destination if exists, then copy entire directory
if os.path.exists(DRIVE_VLLM_DIR):
    shutil.rmtree(DRIVE_VLLM_DIR)

print(f"📤 Copying to Google Drive: {DRIVE_VLLM_DIR}")
shutil.copytree("/content/vllm_export", DRIVE_VLLM_DIR,
                ignore=shutil.ignore_patterns('.cache'))
print(f"\n✅ vLLM model copied to {DRIVE_VLLM_DIR}")

for f in os.listdir(DRIVE_VLLM_DIR):
    size_mb = os.path.getsize(os.path.join(DRIVE_VLLM_DIR, f)) / 1024**2
    print(f"  {f} ({size_mb:.1f} MB)")


## Cell 12 — Push to Hugging Face Hub

In [ ]:
from huggingface_hub import HfApi, login

login(token=HF_TOKEN)
api = HfApi()

phase_suffix = f"-phase{CURRENT_PHASE}"

# 1. Push LoRA adapters + tokenizer
print("📤 Pushing LoRA adapters...")
model.push_to_hub(HF_REPO_ID + phase_suffix + "-lora", token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO_ID + phase_suffix + "-lora", token=HF_TOKEN)
print("✅ LoRA pushed")


In [ ]:
# 2. Push merged 16-bit model (vLLM-compatible)
merged_repo = HF_REPO_ID + phase_suffix
print(f"\n📤 Pushing merged 16-bit model to {merged_repo}...")
api.create_repo(repo_id=merged_repo, exist_ok=True, token=HF_TOKEN)
api.upload_folder(
    folder_path="/content/vllm_export",
    repo_id=merged_repo,
    ignore_patterns=[".cache", ".cache/*"],
    token=HF_TOKEN,
)
print("✅ Merged 16-bit model pushed")
print(f"  Merged: https://huggingface.co/{merged_repo}")


In [ ]:
# 3. Push gguf model (vLLM-compatible)
gguf_repo = HF_REPO_ID + phase_suffix + "-gguf"
print(f"\n📤 Pushing GGUF files to {gguf_repo}...")
api.create_repo(repo_id=gguf_repo, exist_ok=True, token=HF_TOKEN)
api.upload_folder(
    folder_path="/content/gguf_export_gguf",
    repo_id=gguf_repo,
    allow_patterns=["*.gguf", "Modelfile"],
    token=HF_TOKEN,
)
print("✅ GGUF files pushed (Q4_K_M, F16, Q8_0)")
print(f"\n🎉 All models pushed!")
print(f"  GGUF:   https://huggingface.co/{gguf_repo}")
